In [2]:
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_cohere.rerank import CohereRerank
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
chunks = [
    # Tesla - Financial & Production
    "Tesla reported record quarterly revenue of $25.2 billion in Q3 2024.",
    "Tesla's automotive gross margin improved to 19.3% this quarter.",
    "Tesla Cybertruck production ramp begins in 2024 with initial deliveries.",
    "Tesla announced plans to expand Gigafactory production capacity.",
    "Tesla stock price reached new highs following earnings announcement.",
    "Tesla's energy storage business grew 40% year-over-year.",
    "Tesla continues to lead in electric vehicle market share globally.",
    "Tesla Model Y became the best-selling vehicle worldwide.",
    "Tesla reported strong free cash flow generation of $7.5 billion.",
    "Tesla's Full Self-Driving revenue increased significantly.",
    
    # Microsoft - Development & Acquisitions
    "Microsoft acquired GitHub for $7.5 billion in 2018.",
    "Microsoft's cloud revenue Azure grew 29% year-over-year.",
    "Microsoft announced new AI features for Visual Studio Code.",
    "Microsoft Teams integration with GitHub enhances developer workflow.",
    "Microsoft's developer tools division sees strong adoption.",
    "Microsoft acquired Activision Blizzard for $68.7 billion.",
    "Microsoft's productivity suite gained 50 million new users.",
    "Microsoft announced new Surface devices for developers.",
    "Microsoft's AI Copilot features expand to more development tools.",
    "Microsoft's enterprise solutions drive revenue growth.",
    
    # NVIDIA - AI & Hardware
    "NVIDIA's data center revenue reached $47.5 billion annually.",
    "NVIDIA's H100 GPUs see unprecedented demand for AI training.",
    "NVIDIA announced next-generation Blackwell architecture.",
    "NVIDIA's gaming revenue declined due to crypto market changes.",
    "NVIDIA's automotive AI platform partnerships expanded.",
    "NVIDIA's AI chip shortage affects cloud providers.",
    "NVIDIA stock valuation exceeds $2 trillion market cap.",
    "NVIDIA's CUDA platform dominates AI development.",
    "NVIDIA announced new AI inference chips for edge computing.",
    "NVIDIA's partnership with major cloud providers strengthens.",
    
    # Google/Alphabet - AI & Cloud
    "Google's AI investments total over $100 billion in recent years.",
    "Google Cloud revenue grew 35% reaching $8.4 billion quarterly.",
    "Google announced Gemini AI model competing with GPT-4.",
    "Google's search advertising revenue remains strong at $59 billion.",
    "Google's Workspace products integrate advanced AI features.",
    "Google announced quantum computing breakthroughs.",
    "Google's autonomous vehicle division Waymo expands operations.",
    "Google's AI research published breakthrough papers.",
    "Google's cloud AI services see enterprise adoption.",
    "Google faces regulatory scrutiny over AI dominance.",
    
    # Noisy/Less Relevant Chunks
    "The Tesla coil was invented by Nikola Tesla in 1891.",
    "Microsoft Excel spreadsheet formulas can be complex for beginners.",
    "NVIDIA Shield TV streaming device gets software update.",
    "Google Maps navigation improved with real-time traffic data.",
    "Production delays affected multiple manufacturing sectors.",
    "Financial markets showed volatility during earnings season.",
    "Revenue recognition standards changed for software companies.",
    "Hardware components face supply chain constraints globally.",
    "Development tools market grows with remote work trends.",
    "AI research requires significant computational resources.",
    "Quarterly reports show mixed results across tech sector.",
    "Stock market analysts upgrade technology sector ratings.",
    "Cloud computing adoption accelerates in enterprise market.",
    "Data center construction increases globally.",
    "Semiconductor shortage impacts various industries.",
    "Electric vehicle charging infrastructure expands rapidly.",
    "Software development productivity tools gain popularity.",
    "Machine learning frameworks become more accessible.",
    "Enterprise software licensing models evolve.",
    "Technology conferences showcase latest innovations."
]

In [4]:
# Convert to Document objects
documents = [Document(page_content=chunk, metadata={"source": f"chunk_{i}"}) for i, chunk in enumerate(chunks)]

# 1. Vector Retriever

In [5]:
embedding_model = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
vector_store = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    collection_configuration={"hnsw:space": "cosine"}
)
vector_retriever = vector_store.as_retriever(search_kwargs={"k": 15})

/var/folders/44/6s0y2cld6bbg7s1788ts_jj40000gn/T/ipykernel_54846/2961984917.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")


# 2. BM25 - Keyword Retriever

In [6]:
bm25_retriever = BM25Retriever.from_documents(documents)
bm25_retriever.k = 15

# 3. Hybrid Retriever

In [7]:
hybrid_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever],
    weights=[0.7, 0.3]
)

In [8]:
query = "Tesla financial performance and production updates"

print("STEP 1: Hybrid Retrieval Results")
retrieved_docs = hybrid_retriever.invoke(query)

for i, doc in enumerate(retrieved_docs, 1):
    print(f"{i}. {doc.page_content} (Source: {doc.metadata['source']})")

print(f"Retrieved {len(retrieved_docs)} documents for the query.")

STEP 1: Hybrid Retrieval Results
1. Tesla stock price reached new highs following earnings announcement. (Source: chunk_4)
2. Tesla announced plans to expand Gigafactory production capacity. (Source: chunk_3)
3. Tesla reported record quarterly revenue of $25.2 billion in Q3 2024. (Source: chunk_0)
4. Tesla continues to lead in electric vehicle market share globally. (Source: chunk_6)
5. Tesla reported strong free cash flow generation of $7.5 billion. (Source: chunk_8)
6. Tesla Model Y became the best-selling vehicle worldwide. (Source: chunk_7)
7. Tesla Cybertruck production ramp begins in 2024 with initial deliveries. (Source: chunk_2)
8. Financial markets showed volatility during earnings season. (Source: chunk_45)
9. Tesla's automotive gross margin improved to 19.3% this quarter. (Source: chunk_1)
10. Tesla's Full Self-Driving revenue increased significantly. (Source: chunk_9)
11. Tesla's energy storage business grew 40% year-over-year. (Source: chunk_5)
12. Stock market analysts up

In [9]:
print("\nSTEP 2: Reranking with CohereRerank (Top 10)")

reranker = CohereRerank(model="rerank-english-v3.0", top_n=10)

reranked_docs = reranker.compress_documents(retrieved_docs, query)

for i, doc in enumerate(reranked_docs, 1):
    print(f"{i:2d}. {doc.page_content}")


STEP 2: Reranking with CohereRerank (Top 10)
 1. Tesla reported strong free cash flow generation of $7.5 billion.
 2. Tesla reported record quarterly revenue of $25.2 billion in Q3 2024.
 3. Tesla's automotive gross margin improved to 19.3% this quarter.
 4. Tesla announced plans to expand Gigafactory production capacity.
 5. Tesla's energy storage business grew 40% year-over-year.
 6. Tesla continues to lead in electric vehicle market share globally.
 7. Tesla's Full Self-Driving revenue increased significantly.
 8. Tesla Cybertruck production ramp begins in 2024 with initial deliveries.
 9. Tesla stock price reached new highs following earnings announcement.
10. Financial markets showed volatility during earnings season.


In [10]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_groq import ChatGroq
import os

groq_key = os.getenv("GROQ_API_KEY")

print("\n" + "="*80)
print("FINAL: RAG with Reranked Context")
print("-"*40)

# Use top 5 reranked documents for final answer
top_reranked = reranked_docs[:5]

combined_input = f"""Based on the following documents, please answer this question: {query}

Documents:
{chr(10).join([f"- {doc.page_content}" for doc in top_reranked])}

Please provide a clear, helpful answer using only the information from these documents."""

model = ChatGroq(
    model="llama-3.3-70b-versatile",
    groq_api_key=groq_key
)
messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content=combined_input),
]

result = model.invoke(messages)
print("Generated Response:")
print(result.content)


FINAL: RAG with Reranked Context
----------------------------------------
Generated Response:
Based on the provided documents, here are the updates on Tesla's financial performance and production:

1. **Strong Financial Performance**: Tesla reported a record quarterly revenue of $25.2 billion in Q3 2024 and generated $7.5 billion in free cash flow.
2. **Improved Automotive Gross Margin**: The automotive gross margin improved to 19.3% in the quarter.
3. **Energy Storage Business Growth**: The energy storage business saw a 40% year-over-year growth.
4. **Production Capacity Expansion**: Tesla has announced plans to expand Gigafactory production capacity, indicating potential for increased production in the future.

These updates suggest a positive trend in Tesla's financial performance and production capabilities.
